# Prometheus Client & Metrics Instrumentation

In [ ]:
# Day 40 — Prometheus Client & Metrics
# On Day 39 we built a FastAPI model-serving API. Now: how do we MONITOR it?
#
# The 3 Pillars of Observability:
#   1. Logs    -> What happened (Day 44)
#   2. Metrics -> How much / how fast / how often (TODAY!)
#   3. Traces  -> Where time was spent across services
#
# Prometheus scrapes an HTTP `/metrics` endpoint on your Python app.
#
# Today we'll cover:
#   1. Prometheus metric types (Counter, Gauge, Histogram, Summary)
#   2. Instrumenting FastAPI with `prometheus_client`
#   3. Exposing `/metrics` for Prometheus & Grafana
#   4. Custom MLOps metrics (inference latency, prediction class counts)
#
# Install: pip install prometheus-client fastapi uvicorn

### 1. The 4 Prometheus Metric Types

In [ ]:
from prometheus_client import Counter, Gauge, Histogram, generate_latest

# 1. Counter: ONLY goes up (total requests, total errors)
PREDICTIONS_TOTAL = Counter(
    "ml_predictions_total",
    "Total ML predictions served",
    labelnames=["model_version", "status"]
)

# 2. Gauge: goes UP and DOWN (active requests, CPU %, drift score)
MODEL_DRIFT_SCORE = Gauge(
    "ml_data_drift_score",
    "Latest share of drifted features"
)

# 3. Histogram: measures distributions in buckets (latency, payload size)
INFERENCE_LATENCY = Histogram(
    "ml_inference_duration_seconds",
    "Time spent running model.predict()",
    buckets=[0.005, 0.01, 0.025, 0.05, 0.1, 0.5, 1.0]
)

# Simulate recording metrics
PREDICTIONS_TOTAL.labels(model_version="v1", status="healthy").inc()
PREDICTIONS_TOTAL.labels(model_version="v1", status="critical").inc(2)
MODEL_DRIFT_SCORE.set(0.25)
INFERENCE_LATENCY.observe(0.018)

print(generate_latest().decode("utf-8")[:600])

### 2. Instrumenting FastAPI with `/metrics`

In [ ]:
import os
os.makedirs("prom_demo", exist_ok=True)

api_code = '''from fastapi import FastAPI, Response
from pydantic import BaseModel
from prometheus_client import Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST
import time

app = FastAPI(title="Instrumented ML API")

REQ_COUNT = Counter("api_requests_total", "Total API requests", ["endpoint", "status"])
PRED_LATENCY = Histogram("model_predict_seconds", "Model prediction latency")

class ServerInput(BaseModel):
    cpu: float
    memory: float

@app.post("/predict")
def predict(data: ServerInput):
    start = time.perf_counter()
    status = "critical" if data.cpu > 80 else "healthy"
    duration = time.perf_counter() - start

    PRED_LATENCY.observe(duration)
    REQ_COUNT.labels(endpoint="/predict", status=status).inc()
    return {"status": status, "latency_s": round(duration, 6)}

@app.get("/metrics")
def metrics():
    return Response(content=generate_latest(), media_type=CONTENT_TYPE_LATEST)
'''

with open("prom_demo/app_metrics.py", "w") as f:
    f.write(api_code)

print("Created prom_demo/app_metrics.py")
print("Run: uvicorn app_metrics:app --port 8000")
print("Scrape: http://localhost:8000/metrics")

In [ ]:
# Practice Questions:
#
# 1. Why shouldn't you put high-cardinality values (like `user_id` or `timestamp`)
#    into Prometheus labelnames?
# 2. When would you use a Histogram vs a Gauge?
# 3. How does Prometheus collect metrics from your FastAPI pod in Kubernetes?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: FastAPI Model Serving Metrics
# Add a `Counter` for errors (4xx/5xx), a `Histogram` for prediction latency,
# and an `Info` metric exposing `model_version` and `git_sha`.

# Exercise 2: Custom MLOps Metrics Exporter
# Write a standalone script using `start_http_server(9100)` from `prometheus_client`
# that exports `ml_data_drift_score` and `ml_training_duration_seconds` as Gauges.

# Exercise 3: Middleware Timing
# Write a FastAPI HTTP middleware (`@app.middleware("http")`) that automatically
# measures latency for EVERY endpoint without repeating timer code.